# 08. Hybrid search: brute force vs IVF

Searches reviews by meaning plus SQL filters, and compares the exact answer with the faster IVF answer.

## What to look at

- Change only the **QUERY** string (and the filters if you like) in the first cell, then run the rest.
- The **side-by-side tables**: same question, two methods.
- The **stats table**: *files scanned vs total*. IVF opens far fewer files.
- **Recall**: how much of the exact top-k IVF found. Raise NPROBE and run again to watch it climb.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
# ---- change these ----
QUERY  = "my package arrived broken"      # any language works (the model is multilingual)
SINCE  = "2018-01-01"                      # None = no lower date bound
UNTIL  = None
STATUS = "delivered"                       # None = any status
NPROBE = 3                                 # how many clusters IVF looks into (1..50)
TOP_K  = 10

In [ ]:
from embeddings import encode_texts
from search import explain_text, load_centroids, nearest_clusters, search, translations
spark = get_spark("nb08")
C = load_centroids(spark)
q = encode_texts([QUERY])[0]
sims = C @ q
near = nearest_clusters(q, C, NPROBE)
pd.DataFrame({"cluster": near, "cosine(query, centroid)": [round(float(sims[c]), 3) for c in near]})

In [ ]:
kw = dict(since=SINCE, until=UNTIL, status=STATUS, top_k=TOP_K)
search(spark, q, mode="brute", **kw); search(spark, q, mode="ivf", C=C, nprobe=NPROBE, **kw)   # warm-up
brute = search(spark, q, mode="brute", **kw)
ivf = search(spark, q, mode="ivf", C=C, nprobe=NPROBE, **kw)
total_files = spark.sql(f"SELECT count(*) n FROM {T_GOLD}.files").first().n
tr = translations(spark, [r.review_text for r in brute.rows + ivf.rows])   # English lines, only if 05b was run
def table(res):
    d = pd.DataFrame([{"#": i + 1, "score": round(r.score, 3), "cluster": r.cluster_id, "date": f"{r.purchase_ts:%Y-%m-%d}",
                       "status": r.order_status, "stars": r.review_score, "review": r.review_text[:110],
                       **({"english": tr.get(r.review_text, "")[:110]} if tr else {})} for i, r in enumerate(res.rows)])
    return d
side_by_side([table(ivf), table(brute)], [f"IVF (nprobe={NPROBE})", "Brute force (exact)"])

In [ ]:
def stats(name, res):
    m = res.metrics
    return {"method": name, "latency ms": round(res.seconds * 1000), "files scanned": m.get("resultDataFiles"), "files total": total_files,
            "% of files": round(100 * m.get("resultDataFiles", 0) / total_files, 1), "rows read": m.get("numOutputRows")}
pd.DataFrame([stats(f"IVF nprobe={NPROBE}", ivf), stats("Brute force", brute)])

In [ ]:
truth = [(r.review_id, r.order_id) for r in brute.rows]
got = {(r.review_id, r.order_id) for r in ivf.rows}
floor = brute.rows[-1].score if brute.rows else float("-inf")
tie = sum(1 for r in ivf.rows if r.score >= floor - 1e-4)
n = len(truth) or 1
pd.DataFrame([("recall@k by review", round(len(got & set(truth)) / n, 3)),
              ("recall@k tie-aware (identical reviews are interchangeable)", round(min(tie, n) / n, 3))], columns=["metric", "value"])

In [ ]:
# the pushed-down predicates: this is where cluster_id IN (...) shows up
print("\n".join(l for l in explain_text(ivf.df).splitlines() if "filters=" in l))

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Nothing is written by a search; it only reads. To see what it read, open `gold/review_embeddings/data/` and compare: brute force touches every month folder in the date range and all files in them, IVF only the few files whose cluster range contains the probed clusters (the *files scanned* number above).